# March Madness Prediction Project

This notebook runs the full pipeline using functions and classes defined in
`marchmadness.py`. All scraping, data processing, modeling, and plotting
logic lives in that module — this notebook only calls into it.

**Pipeline overview:**
1. Scrape KenPom + Barttorvik regular-season stats (Selection Sunday cutoff)
2. Process Kaggle tournament results
3. Build mega/model datasets with difference features
4. Seed-only baselines
5. Train Logistic Regression, Random Forest, Gradient Boosting, Neural Network
6. Visualize and interpret results


## Setup

In [ ]:
# !pip install rapidfuzz  # uncomment if needed

In [ ]:
import marchmadness as mm

## Part 1: KenPom & Barttorvik Webscraping

Scrapes both sites for each season (cutoff = Selection Sunday, to avoid
tournament data leakage), fuzzy-matches team names, and saves
`RegSeasonStats.csv`.

⚠️ This makes many web requests and takes ~5 minutes.

In [ ]:
reg_season_stats = mm.build_regular_season_stats()
reg_season_stats.head()

## Part 2: Tournament Win/Loss Records

Loads the Kaggle `MMHistoricResults.csv` and `MTeams.csv` files and attaches
team names to each historical tournament game.

> NOTE: Download `MMHistoricResults.csv` and `MTeams.csv` from the Kaggle
> 2026 March Madness ML competition and place them in this directory.

In [ ]:
games = mm.build_tournament_results()
games.head()

## Part 3: Building the Mega & Model Datasets

Merges regular-season stats onto each tournament game to produce:
- `MarchMadness_MegaDataset_2010_2025.csv` (winner/loser framing)
- `MarchMadness_ModelDataset_2010_2025.csv` (Team1/Team2 framing with diff features)

In [ ]:
mega_df, model_df = mm.build_datasets()

## Part 4: Models

### Seed-only baselines
- Logistic regression on seed difference
- Deterministic "lower seed wins" rule
- Empirical historical seed-matchup win rates

In [ ]:
seed_data = mm.prepare_seed_data()
baseline_results = mm.run_seed_baselines(seed_data)

print(f"Seed Only Logistic Regression Accuracy: {baseline_results['seed_logreg']:.2%}")
print(f"Lower-seed-wins baseline:               {baseline_results['lower_seed_wins']:.2%}")
print(f"Empirical matchup baseline:             {baseline_results['empirical_matchup']:.2%}")

### Feature-based models

Uses all `_diff` features (excluding seed) to predict `Team1Win`.

In [ ]:
df = mm.pd.read_csv('MarchMadness_ModelDataset_2010_2025.csv')
feature_cols = mm.get_feature_columns(df)
X_train, X_test, y_train, y_test = mm.split_features(df, feature_cols)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

#### Logistic Regression

In [ ]:
log_model, log_acc, log_coefs = mm.train_logistic_regression(X_train, y_train, X_test, y_test)
print(f"Logistic Regression prediction accuracy: {log_acc:.2%}")
log_coefs.head(20)

#### Random Forest

In [ ]:
rf_model, rf_acc, rf_train_acc, rf_importances = mm.train_random_forest(X_train, y_train, X_test, y_test)
print(f"Random Forest train accuracy: {rf_train_acc:.2%}")
print(f"Random Forest test accuracy:  {rf_acc:.2%}")
rf_importances.head(10)

#### Gradient Boosting

In [ ]:
gb_model, gb_acc, gb_train_acc, gb_auc, gb_importances = mm.train_gradient_boosting(X_train, y_train, X_test, y_test)
print(f"Gradient Boosting train accuracy: {gb_train_acc:.2%}")
print(f"Gradient Boosting test accuracy:  {gb_acc:.2%}")
print(f"Gradient Boosting AUC: {gb_auc:.3f}")
gb_importances.head(10)

#### Neural Network

In [ ]:
nn_model, nn_scaler, nn_acc, nn_probs, nn_epoch_accs = mm.train_neural_network(X_train, y_train, X_test, y_test)
print(f"\nNeural Network prediction accuracy: {nn_acc:.2%}")

## Part 5: Visualizing and Interpreting Models

In [ ]:
log_pred = log_model.predict(X_test)
rf_pred = rf_model.predict(X_test)
gb_pred = gb_model.predict(X_test)
nn_pred = (nn_probs > 0.5).astype(int)

model_results = [
    {"Model": "Empirical Matchup", "Accuracy": baseline_results['empirical_matchup']},
    {"Model": "Logistic Regression", "Accuracy": log_acc},
    {"Model": "Random Forest", "Accuracy": rf_acc},
    {"Model": "Gradient Boosting", "Accuracy": gb_acc},
    {"Model": "Neural Network", "Accuracy": nn_acc},
]

accuracy_plot = mm.plot_accuracy_comparison(model_results, baseline_line=baseline_results['lower_seed_wins'])

### Permutation Importance

In [ ]:
models = {
    "Logistic Regression": log_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
}
top_feature_table = mm.plot_permutation_importance(models, X_test, y_test)
top_feature_table

### Confusion Matrices

In [ ]:
preds = {
    "Logistic Regression": log_pred,
    "Random Forest": rf_pred,
    "Gradient Boosting": gb_pred,
    "Neural Network": nn_pred,
}
mm.plot_confusion_matrices(preds, y_test)

### ROC Curves

In [ ]:
roc_models = {
    "Logistic Regression": log_model.predict_proba(X_test)[:, 1],
    "Random Forest": rf_model.predict_proba(X_test)[:, 1],
    "Gradient Boosting": gb_model.predict_proba(X_test)[:, 1],
    "Neural Network": nn_probs.ravel(),
}
mm.plot_roc_curves(roc_models, y_test)

### Predicted Win Probability Distributions

In [ ]:
mm.plot_probability_distributions(roc_models, y_test)

### Logistic Regression Coefficients

In [ ]:
mm.plot_logistic_coefficients(log_model, feature_cols)